# NLU Comparison: GPT-4o-mini vs Gemini 2.5 Flash

Compara Phase A (intent detection) entre OpenAI y Google Gemini.
Mismo prompt, mismos test cases, lado a lado.

In [9]:
# Instalar dependencias si no las tienes
!pip install openai google-genai -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import time
from openai import OpenAI
from google import genai
from google.genai import types

# ====== KEYS ======
OPENAI_KEY = "REDACTED_OPENAI_KEY"
GEMINI_KEY = "REDACTED_GEMINI_KEY"

OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash-lite"

openai_client = OpenAI(api_key=OPENAI_KEY)
gemini_client = genai.Client(api_key=GEMINI_KEY)

print(f"OpenAI: {OPENAI_MODEL}")
print(f"Gemini: {GEMINI_MODEL}")
print("Clients ready.")

In [10]:
# ====== SYSTEM PROMPT (Phase A simplificado para testing) ======

CATEGORIES = "Alimentación, Transporte, Hogar, Salud, Personal, Entretenimiento, Educación, Servicios"

SYSTEM_PROMPT = f"""Eres Gus, asistente financiero personal. Ayudas a usuarios a gestionar sus gastos.

Tu tarea es analizar el mensaje del usuario y decidir:
1. Si necesitas usar una herramienta (tool_call)
2. Si necesitas mas informacion (clarification)
3. Si puedes responder directamente (direct_reply) — Para saludos O temas fuera del dominio

=== HERRAMIENTAS DISPONIBLES ===
- register_transaction: Registra gasto/ingreso. Args: amount (number, CLP), category (string), name (string 2-4 palabras), type ("expense" o "income")
- ask_balance: Consulta saldo. Sin args.
- ask_budget_status: Estado del presupuesto. Sin args.
- ask_goal_status: Progreso de metas. Sin args.
- ask_app_info: Info sobre el bot/app. Args: userQuestion (string)
- greeting: Saludos simples. Sin args.
- manage_transactions: Gestionar transacciones. Args: operation ("list"/"edit"/"delete")
- manage_categories: Gestionar categorias. Args: operation ("list"/"create"/"rename"/"delete"), name

=== CATEGORIAS DEL USUARIO ===
{CATEGORIES}

=== REGLAS ===
- "lucas" = x1000 CLP ("15 lucas" = 15000)
- DEDUCE categoria del contexto (uber → Transporte, restaurante → Alimentación)
- Si no hay monto explícito → clarification
- Saludos simples → direct_reply
- Temas fuera de finanzas → direct_reply con redirect amable
- SIEMPRE deduce un name corto (2-4 palabras)
- Ingresos: "me pagaron", "recibí sueldo" → type: "income"

=== FORMATO JSON (estricto) ===
response_type: "tool_call" | "clarification" | "direct_reply"

Ejemplos:
{{"response_type": "tool_call", "tool_call": {{"name": "register_transaction", "args": {{"amount": 15000, "category": "Alimentación", "name": "Almuerzo"}}}}}}
{{"response_type": "clarification", "clarification": "¿Cuánto gastaste?"}}
{{"response_type": "direct_reply", "direct_reply": "¡Hola! ¿En qué te ayudo?"}}
"""

print(f"Prompt: {len(SYSTEM_PROMPT)} chars")

Prompt: 1865 chars


In [11]:
# ====== FUNCIONES DE LLAMADA ======

def call_openai(user_text: str) -> dict:
    """Llama a GPT-4o-mini con JSON mode."""
    start = time.time()
    completion = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
        timeout=25.0,
    )
    elapsed = round((time.time() - start) * 1000)
    raw = completion.choices[0].message.content or "{}"
    usage = completion.usage
    return {
        "provider": "openai",
        "model": OPENAI_MODEL,
        "response": json.loads(raw),
        "elapsed_ms": elapsed,
        "tokens_in": usage.prompt_tokens if usage else 0,
        "tokens_out": usage.completion_tokens if usage else 0,
    }


def call_gemini(user_text: str) -> dict:
    """Llama a Gemini Flash con JSON mode."""
    start = time.time()
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[types.Content(role="user", parts=[types.Part.from_text(text=user_text)])],
        config=types.GenerateContentConfig(
            temperature=0.3,
            response_mime_type="application/json",
            system_instruction=SYSTEM_PROMPT,
        ),
    )
    elapsed = round((time.time() - start) * 1000)
    raw = (response.text or "{}").strip()
    usage = response.usage_metadata
    return {
        "provider": "gemini",
        "model": GEMINI_MODEL,
        "response": json.loads(raw),
        "elapsed_ms": elapsed,
        "tokens_in": usage.prompt_token_count if usage else 0,
        "tokens_out": usage.candidates_token_count if usage else 0,
    }


# Quick test
print("Testing OpenAI...")
r1 = call_openai("hola")
print(f"  ✅ {r1['elapsed_ms']}ms — {r1['response']}")

print("Testing Gemini...")
r2 = call_gemini("hola")
print(f"  ✅ {r2['elapsed_ms']}ms — {r2['response']}")

Testing OpenAI...
  ✅ 2441ms — {'response_type': 'direct_reply', 'direct_reply': '¡Hola! ¿En qué te ayudo?'}
Testing Gemini...
  ✅ 2544ms — {'response_type': 'direct_reply', 'direct_reply': '¡Hola! ¿En qué puedo ayudarte hoy?'}


In [12]:
# ====== TEST CASES ======

TEST_CASES = [
    # (label, input_text, expected_type, expected_tool)
    
    # --- Registro de gastos ---
    ("Gasto simple", "gasté 15 lucas en comida", "tool_call", "register_transaction"),
    ("Gasto con deducción", "tomé un uber al trabajo, 5 lucas", "tool_call", "register_transaction"),
    ("Gasto farmacia", "pagué 8 lucas en la farmacia", "tool_call", "register_transaction"),
    ("Gasto arriendo", "pagué el arriendo, 450 lucas", "tool_call", "register_transaction"),
    ("Gasto sin categoría explícita", "gasté 3 lucas en la pelu", "tool_call", "register_transaction"),
    ("Gasto sin monto", "compré una bebida", "clarification", None),
    
    # --- Ingresos ---
    ("Ingreso sueldo", "me pagaron el sueldo, 800 lucas", "tool_call", "register_transaction"),
    ("Ingreso venta", "vendí un mueble en 50 lucas", "tool_call", "register_transaction"),
    
    # --- Consultas ---
    ("Balance", "cuánto me queda?", "tool_call", "ask_balance"),
    ("Presupuesto", "cómo voy con mi presupuesto?", "tool_call", "ask_budget_status"),
    ("Metas", "cuánto me falta para mi meta?", "tool_call", "ask_goal_status"),
    
    # --- App info ---
    ("Nombre del bot", "cómo te llamas?", "tool_call", "ask_app_info"),
    ("Capacidades", "qué puedes hacer?", "tool_call", "ask_app_info"),
    
    # --- Saludos ---
    ("Saludo", "hola", "direct_reply", None),
    ("Buenos días", "buenos días", "direct_reply", None),
    
    # --- Fuera de dominio ---
    ("Matemáticas", "cuál es la raíz cuadrada de 144?", "direct_reply", None),
    ("Historia", "quién fue Napoleón?", "direct_reply", None),
    
    # --- Gestión ---
    ("Listar gastos", "mis últimos gastos", "tool_call", "manage_transactions"),
    ("Listar categorías", "qué categorías tengo?", "tool_call", "manage_categories"),
    ("Crear categoría", "crea la categoría Mascotas", "tool_call", "manage_categories"),
    
    # --- Chilenismos ---
    ("Lucas en super", "10 lucas en el super", "tool_call", "register_transaction"),
    ("Coloquial", "weón gasté 20 lucas en copete", "tool_call", "register_transaction"),
]

print(f"{len(TEST_CASES)} test cases listos.")

22 test cases listos.


In [13]:
# ====== EJECUTAR COMPARACIÓN ======

results = []

for i, (label, text, exp_type, exp_tool) in enumerate(TEST_CASES):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(TEST_CASES)}] {label}")
    print(f"Input: \"{text}\"")
    print(f"Expected: type={exp_type}{f', tool={exp_tool}' if exp_tool else ''}")
    print(f"{'─'*60}")
    
    # OpenAI
    try:
        r_oai = call_openai(text)
        oai_type = r_oai["response"].get("response_type", "?")
        oai_tool = r_oai["response"].get("tool_call", {}).get("name", "—") if isinstance(r_oai["response"].get("tool_call"), dict) else "—"
        oai_args = r_oai["response"].get("tool_call", {}).get("args", {}) if isinstance(r_oai["response"].get("tool_call"), dict) else {}
        oai_ok = oai_type == exp_type and (exp_tool is None or oai_tool == exp_tool)
        icon_oai = "✅" if oai_ok else "❌"
        print(f"  {icon_oai} OpenAI  ({r_oai['elapsed_ms']:>4}ms, {r_oai['tokens_in']}+{r_oai['tokens_out']} tok) → type={oai_type}, tool={oai_tool}")
        if oai_args:
            print(f"     args={json.dumps(oai_args, ensure_ascii=False)}")
    except Exception as e:
        r_oai = {"error": str(e)}
        oai_type = oai_tool = "ERROR"
        oai_ok = False
        print(f"  ❌ OpenAI ERROR: {e}")
    
    # Gemini
    try:
        r_gem = call_gemini(text)
        gem_type = r_gem["response"].get("response_type", "?")
        gem_tool = r_gem["response"].get("tool_call", {}).get("name", "—") if isinstance(r_gem["response"].get("tool_call"), dict) else "—"
        gem_args = r_gem["response"].get("tool_call", {}).get("args", {}) if isinstance(r_gem["response"].get("tool_call"), dict) else {}
        gem_ok = gem_type == exp_type and (exp_tool is None or gem_tool == exp_tool)
        icon_gem = "✅" if gem_ok else "❌"
        print(f"  {icon_gem} Gemini  ({r_gem['elapsed_ms']:>4}ms, {r_gem['tokens_in']}+{r_gem['tokens_out']} tok) → type={gem_type}, tool={gem_tool}")
        if gem_args:
            print(f"     args={json.dumps(gem_args, ensure_ascii=False)}")
    except Exception as e:
        r_gem = {"error": str(e)}
        gem_type = gem_tool = "ERROR"
        gem_ok = False
        print(f"  ❌ Gemini ERROR: {e}")
    
    # Match?
    match = oai_type == gem_type and oai_tool == gem_tool
    print(f"  {'🟢' if match else '🔴'} {'Coinciden' if match else 'DIFIEREN'}")
    
    results.append({
        "label": label,
        "text": text,
        "expected_type": exp_type,
        "expected_tool": exp_tool,
        "openai": r_oai,
        "gemini": r_gem,
        "openai_correct": oai_ok,
        "gemini_correct": gem_ok,
        "providers_match": match,
    })
    
    # Rate limit: Gemini free tier = 10 RPM
    time.sleep(7)


[1/22] Gasto simple
Input: "gasté 15 lucas en comida"
Expected: type=tool_call, tool=register_transaction
────────────────────────────────────────────────────────────
  ✅ OpenAI  (2668ms, 516+51 tok) → type=tool_call, tool=register_transaction
     args={"amount": 15000, "category": "Alimentación", "name": "Comida", "type": "expense"}
  ✅ Gemini  (2342ms, 549+58 tok) → type=tool_call, tool=register_transaction
     args={"amount": 15000, "category": "Alimentación", "name": "Comida", "type": "expense"}
  🟢 Coinciden

[2/22] Gasto con deducción
Input: "tomé un uber al trabajo, 5 lucas"
Expected: type=tool_call, tool=register_transaction
────────────────────────────────────────────────────────────
  ✅ OpenAI  (4705ms, 519+51 tok) → type=tool_call, tool=register_transaction
     args={"amount": 5000, "category": "Transporte", "name": "Uber al trabajo", "type": "expense"}
  ✅ Gemini  (2026ms, 551+57 tok) → type=tool_call, tool=register_transaction
     args={"amount": 5000, "category": "Tr

In [14]:
# ====== RESUMEN ======

oai_correct = sum(1 for r in results if r["openai_correct"])
gem_correct = sum(1 for r in results if r["gemini_correct"])
both_match = sum(1 for r in results if r["providers_match"])
total = len(results)

oai_times = [r["openai"]["elapsed_ms"] for r in results if "elapsed_ms" in r.get("openai", {})]
gem_times = [r["gemini"]["elapsed_ms"] for r in results if "elapsed_ms" in r.get("gemini", {})]

oai_tokens_in = sum(r["openai"].get("tokens_in", 0) for r in results if isinstance(r.get("openai"), dict) and "tokens_in" in r["openai"])
oai_tokens_out = sum(r["openai"].get("tokens_out", 0) for r in results if isinstance(r.get("openai"), dict) and "tokens_out" in r["openai"])
gem_tokens_in = sum(r["gemini"].get("tokens_in", 0) for r in results if isinstance(r.get("gemini"), dict) and "tokens_in" in r["gemini"])
gem_tokens_out = sum(r["gemini"].get("tokens_out", 0) for r in results if isinstance(r.get("gemini"), dict) and "tokens_out" in r["gemini"])

# Pricing
oai_cost = (oai_tokens_in * 0.15 / 1_000_000) + (oai_tokens_out * 0.60 / 1_000_000)
gem_cost = (gem_tokens_in * 0.30 / 1_000_000) + (gem_tokens_out * 2.50 / 1_000_000)

print("\n" + "=" * 60)
print("RESUMEN COMPARACIÓN NLU")
print("=" * 60)
print(f"\nTotal tests: {total}")
print(f"")
print(f"{'Métrica':<25} {'OpenAI':>12} {'Gemini':>12}")
print(f"{'─'*25} {'─'*12} {'─'*12}")
print(f"{'Correctos':<25} {f'{oai_correct}/{total}':>12} {f'{gem_correct}/{total}':>12}")
print(f"{'Accuracy':<25} {f'{oai_correct/total*100:.0f}%':>12} {f'{gem_correct/total*100:.0f}%':>12}")
print(f"{'Latencia promedio':<25} {f'{sum(oai_times)//len(oai_times)}ms':>12} {f'{sum(gem_times)//len(gem_times)}ms':>12}")
print(f"{'Latencia min':<25} {f'{min(oai_times)}ms':>12} {f'{min(gem_times)}ms':>12}")
print(f"{'Latencia max':<25} {f'{max(oai_times)}ms':>12} {f'{max(gem_times)}ms':>12}")
print(f"{'Tokens (in+out)':<25} {f'{oai_tokens_in}+{oai_tokens_out}':>12} {f'{gem_tokens_in}+{gem_tokens_out}':>12}")
print(f"{'Costo total':<25} {f'${oai_cost:.5f}':>12} {f'${gem_cost:.5f}':>12}")
print(f"{'Costo/request prom':<25} {f'${oai_cost/total:.6f}':>12} {f'${gem_cost/total:.6f}':>12}")
print(f"")
print(f"Coincidencias entre providers: {both_match}/{total} ({both_match/total*100:.0f}%)")
print(f"")

# Diferencias
diffs = [r for r in results if not r["providers_match"]]
if diffs:
    print(f"\n{'='*60}")
    print(f"DIFERENCIAS ({len(diffs)})")
    print(f"{'='*60}")
    for r in diffs:
        oai_r = r['openai'].get('response', {})
        gem_r = r['gemini'].get('response', {})
        print(f"\n  {r['label']}: \"{r['text']}\"")
        print(f"    OpenAI: type={oai_r.get('response_type','?')}, tool={oai_r.get('tool_call',{}).get('name','—') if isinstance(oai_r.get('tool_call'), dict) else '—'}")
        print(f"    Gemini: type={gem_r.get('response_type','?')}, tool={gem_r.get('tool_call',{}).get('name','—') if isinstance(gem_r.get('tool_call'), dict) else '—'}")
else:
    print("🎉 Ambos providers dieron exactamente los mismos resultados!")


RESUMEN COMPARACIÓN NLU

Total tests: 22

Métrica                         OpenAI       Gemini
───────────────────────── ──────────── ────────────
Correctos                        19/22        21/22
Accuracy                           86%          95%
Latencia promedio               1932ms       2118ms
Latencia min                    1088ms       1732ms
Latencia max                    4705ms       2564ms
Tokens (in+out)              11330+862    11493+945
Costo total                   $0.00222     $0.00581
Costo/request prom           $0.000101    $0.000264

Coincidencias entre providers: 18/22 (82%)


DIFERENCIAS (4)

  Gasto sin categoría explícita: "gasté 3 lucas en la pelu"
    OpenAI: type=tool_call, tool=register_transaction
    Gemini: type=?, tool=—

  Nombre del bot: "cómo te llamas?"
    OpenAI: type=direct_reply, tool=—
    Gemini: type=tool_call, tool=ask_app_info

  Capacidades: "qué puedes hacer?"
    OpenAI: type=direct_reply, tool=—
    Gemini: type=tool_call, tool=ask_a

In [8]:
# ====== TEST INDIVIDUAL (para probar mensajes específicos) ======

def test_one(text: str):
    """Prueba un mensaje con ambos providers."""
    print(f"\nInput: \"{text}\"")
    print(f"{'─'*50}")
    
    r1 = call_openai(text)
    r2 = call_gemini(text)
    
    print(f"OpenAI  ({r1['elapsed_ms']:>4}ms): {json.dumps(r1['response'], ensure_ascii=False, indent=2)}")
    print(f"Gemini  ({r2['elapsed_ms']:>4}ms): {json.dumps(r2['response'], ensure_ascii=False, indent=2)}")
    
    match = r1['response'].get('response_type') == r2['response'].get('response_type')
    print(f"\n{'🟢 Coinciden' if match else '🔴 Difieren'} en response_type")


# Prueba lo que quieras:
test_one("gasté 15 lucas en un almuerzo con amigos")


Input: "gasté 15 lucas en un almuerzo con amigos"
──────────────────────────────────────────────────


KeyboardInterrupt: 